# MetaJudge: Confidence Calibration Benchmark (Lean)
**Track:** Metacognition — Measuring Progress Toward AGI  
**Scoring:** 1 − per-item Brier loss (strictly proper)  
**Items:** 102-item V4 calibration set  
**Scoring logic:** `metajudge/scoring/adjudication.py`

In [ ]:
# Cell 1 — Imports & Setup
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import json, os

# Import scoring from package
# NOTE: On Kaggle, metajudge/ must be available as a utility script or dataset.
from metajudge.scoring.adjudication import adjudicate, brier_item_score, set_answer_key
from metajudge.utils.text import normalize_text

print(f"SDK imported: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print("Environment OK")

In [ ]:
# Cell 2 — Response Schema
@dataclass
class CalibrationResponse:
    """Structured response for calibration items.

    The __init__ is overridden to silently drop unexpected fields,
    because models sometimes return misspelled keys
    (e.g., 'reason_for_uncertainity' instead of 'reason_for_uncertainty').
    """
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        # Apply defaults for any missing fields
        for name, field in self.__dataclass_fields__.items():
            if not hasattr(self, name):
                setattr(self, name, field.default)

In [ ]:
# Cell 3 — Load Dataset & Answer Key
import pandas as pd

with open('data/metajudge_benchmark_v1.json') as f:
    _items = json.load(f)

dataset = pd.DataFrame(_items)

# Build and register the answer key
ANSWER_KEY = {
    item['item_id']: {k: item[k] for k in ('gold_answer', 'aliases', 'rule')}
    for item in _items
}
set_answer_key(ANSWER_KEY)

print(f"Dataset loaded: {len(dataset)} items")
print(f"Answer key: {len(ANSWER_KEY)} entries")
assert len(ANSWER_KEY) == 102, f"Expected 102, got {len(ANSWER_KEY)}"

In [ ]:
# Cell 4 — Scoring Self-Test
assert adjudicate('gen_b3_024', '3', '3') == True,                      "bare numeric"
assert adjudicate('gen_b3_024', 'three', '3') == True,                  "word-to-digit"
assert adjudicate('gen_b3_024', 'An octopus has three hearts', '3') == True, "verbose numeric"
assert adjudicate('gen_a3_032', '299,792,458', '299792458') == True,    "comma-formatted"
assert adjudicate('gen_b_040', 'contested', 'contested') == True,       "contested match"
assert adjudicate('gen_a4_001', 'the claim is false', 'false') == True, "wrapper stripping"
assert adjudicate('gen_b2_021', 'the answer is france', 'france') == True, "alias wrapper"
assert adjudicate('gen_b3_024', 'probably 3', '3') == False,            "reject hedging"
assert adjudicate('gen_b2_021', 'france because it has the most visitors', 'france') == False, "reject explanation"
print("Scoring self-tests passed")

In [ ]:
# Cell 5 — Benchmark Task Definition

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str, mechanism_primary: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation."""
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    is_correct = adjudicate(item_id, response.answer, gold_answer)
    score = brier_item_score(is_correct, conf)

    print(f"  [{item_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score


# Smoke test
print("=== Smoke test (single item) ===")
_smoke = dataset.iloc[0]
result = metacog_calibration.run(
    llm=kbench.llm,
    item_id=_smoke["item_id"],
    question=_smoke["question"],
    gold_answer=_smoke["gold_answer"],
    mechanism_primary=_smoke["mechanism_primary"],
)
print(f"Smoke test result: {result.result}")

In [ ]:
# Cell 6 — Batch Evaluation (full 102-item V4 calibration set)

@kbench.task(name="metacog_calibration_v1_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run the full 102-item V4 calibration benchmark across all items.

    Returns the headline score: mean of per-item Brier-derived scores.
    This is the official MetaJudge calibration metric.
    """
    eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
    eval_df_input = df[eval_cols].copy()

    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == eval_df_input.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=eval_df_input,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)
    headline = float(scores.mean())

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results")
    print(f"{'='*50}")
    print(f"Items evaluated: {len(scores)}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{float(scores.min()):.4f}, {float(scores.max()):.4f}]")
    print(f"{'='*50}")

    return headline

headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score.result}")

In [ ]:
# Cell 7 — Multi-Model Sweep

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

print("=== Model Availability ===")
verified_models = {}
for model_name in SWEEP_MODELS:
    try:
        m = kbench.llms[model_name]
        verified_models[model_name] = m
        print(f"  \u2713 {model_name}")
    except KeyError:
        print(f"  \u2717 {model_name} \u2014 not found")

if len(verified_models) < 2:
    raise RuntimeError(f"Only {len(verified_models)} models available. Need \u22652.")

print(f"\n{len(verified_models)}/{len(SWEEP_MODELS)} models verified.\n")

eval_cols = ["item_id", "question", "gold_answer", "mechanism_primary"]
eval_df_sweep = dataset[eval_cols].copy()

all_sweep_dfs = []

for model_name, model_obj in verified_models.items():
    print(f"\n{'='*60}")
    print(f"  SWEEPING: {model_name}")
    print(f"{'='*60}")

    max_retries = 3
    for attempt in range(1, max_retries + 1):
        try:
            with kbench.client.enable_cache():
                model_runs = metacog_calibration.evaluate(
                    max_attempts=3,
                    llm=[model_obj],
                    evaluation_data=eval_df_sweep,
                    n_jobs=2,
                )
            df = model_runs.as_dataframe()
            df["model"] = model_name
            all_sweep_dfs.append(df)
            print(f"\n  \u2713 {model_name}: {len(df)} rows collected")
            break
        except Exception as e:
            print(f"\n  \u26a0 Attempt {attempt}/{max_retries} failed: {e}")
            if attempt < max_retries:
                import time
                wait = 30 * attempt
                print(f"    Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  \u2717 {model_name}: all retries exhausted, skipping")

import pandas as pd
if all_sweep_dfs:
    sweep_df = pd.concat(all_sweep_dfs, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"SWEEP COMPLETE: {len(all_sweep_dfs)}/{len(verified_models)} models, {len(sweep_df)} rows")
    print(f"{'='*60}")
else:
    print("\nNo models completed successfully.")

In [ ]:
%choose metacog_calibration_v1_batch